# 1. Datos sintéticos y consolidado tipado

Demo local sin credenciales bancarias. Faker genera 30 mil clientes ficticios, cuentas, movimientos, finanzas y préstamos. DuckDB ejecuta SQL compilado desde la DSL; los arrays y objetos se conservan como `STRUCT`/`LIST`, no como cadenas JSON.

In [1]:
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Ejecutar desde el proyecto o notebooks/"
import duckdb
from rule_manager.synthetic import generate_banking_data, ALLOWED_TABLES
from rule_manager.duckdb_backend import DuckDBBackend
from rule_manager.examples import input_definition
from rule_manager.inputs import compile_inputs
ROOT.joinpath("data").mkdir(exist_ok=True)
con = duckdb.connect(str(ROOT / "data/banking.duckdb"))
if not con.execute("SELECT count(*) FROM information_schema.tables WHERE table_name='synthetic_manifest'").fetchone()[0]:
    print(generate_banking_data(con, clients=30000, seed=20260909))
else:
    assert con.execute("SELECT seed, clients FROM synthetic_manifest").fetchone() == (20260909, 30000)
    print("Fixture sintético existente; no se reemplazan fuentes.")
backend = DuckDBBackend(con, ALLOWED_TABLES)


Fixture sintético existente; no se reemplazan fuentes.


## Modelo relacional

`clients → finances` y `clients.spouse_id → clients → finances` son relaciones 1:1. `clients → accounts → movements` y `clients → loans` son relaciones 1:N. El cliente C000002 demuestra dos cuentas, tres movimientos por cuenta y tres préstamos sin multiplicación de filas.

In [2]:
{name: con.execute(f"SELECT count(*) FROM {name}").fetchone()[0] for name in ALLOWED_TABLES}

{'clients': 30000,
 'finances': 30000,
 'accounts': 44926,
 'movements': 67244,
 'loans': 29894}

In [3]:
inputs = input_definition()
plan = compile_inputs(inputs, backend.catalog())
print(plan.sql)


SELECT CAST("c"."client_id" AS VARCHAR) AS client, CAST(struct_pack("name" := "c"."name", "age" := "c"."age", "financial" := (SELECT struct_pack("income" := "f"."income", "debt" := "f"."debt") FROM "finances" AS "f" WHERE "f"."client_id" = "c"."client_id"), "spouse" := (SELECT struct_pack("name" := "sp"."name", "financial" := (SELECT struct_pack("income" := "sf"."income") FROM "finances" AS "sf" WHERE "sf"."client_id" = "sp"."client_id")) FROM "clients" AS "sp" WHERE "sp"."client_id" = "c"."spouse_id"), "accounts" := (SELECT coalesce(list(struct_pack("id" := "a"."account_id", "balance" := "a"."balance", "kind" := "a"."kind", "movements" := (SELECT coalesce(list(struct_pack("id" := "m"."movement_id", "amount" := "m"."amount", "date" := "m"."movement_date") ORDER BY "m"."movement_id"), []::STRUCT("id" VARCHAR, "amount" DECIMAL(18,2), "date" DATE)[]) FROM "movements" AS "m" WHERE "m"."account_id" = "a"."account_id")) ORDER BY "a"."account_id"), []::STRUCT("id" VARCHAR, "balance" DECIMAL(1

In [4]:
report = backend.materialize(inputs, "input-demo-001", "2026-09-09")
assert report["rows"] == 30000
print({k:v for k,v in report.items() if k != "details"})
con.execute("DESCRIBE consolidated_inputs").fetchall()


{'execution_id': 'input-demo-001', 'status': 'completed', 'rows': 30000, 'reused': True}


[('execution_id', 'VARCHAR', 'NO', 'PRI', None, None),
 ('date', 'DATE', 'NO', 'PRI', None, None),
 ('client', 'VARCHAR', 'NO', 'PRI', None, None),
 ('version', 'BIGINT', 'NO', 'PRI', None, None),
 ('created_by', 'VARCHAR', 'YES', None, None, None),
 ('fields',
  'STRUCT("name" VARCHAR, age BIGINT, financial STRUCT(income DECIMAL(18,2), debt DECIMAL(18,2)), spouse STRUCT("name" VARCHAR, financial STRUCT(income DECIMAL(18,2))), accounts STRUCT(id VARCHAR, balance DECIMAL(18,2), kind VARCHAR, movements STRUCT(id VARCHAR, amount DECIMAL(18,2), date DATE)[])[], loans STRUCT(id VARCHAR, outstanding DECIMAL(18,2))[])',
  'YES',
  None,
  None,
  None)]

In [5]:
fields = con.execute("SELECT fields FROM consolidated_inputs WHERE client='C000002'").fetchone()[0]
assert len(fields["accounts"]) == 2
assert len(fields["accounts"][0]["movements"]) == 3
assert len(fields["loans"]) == 3
assert con.execute("SELECT fields.accounts FROM consolidated_inputs WHERE client='C000001'").fetchone()[0] == []
print("Arrays independientes y anidados verificados.")
con.execute("SELECT client, fields.age, len(fields.accounts), len(fields.loans) FROM consolidated_inputs ORDER BY client LIMIT 5").fetchall()


Arrays independientes y anidados verificados.


[('C000000', 40, 0, 0),
 ('C000001', 40, 0, 0),
 ('C000002', 40, 2, 3),
 ('C000003', 40, 3, 2),
 ('C000004', 40, 2, 0)]

DuckDB no ofrece particiones físicas en su DDL de tabla. La clave de ejecución se conserva en la tabla y la exportación Parquet usa carpetas particionadas por `execution_id`, como prueba local del contrato.

In [6]:
backend.export_inputs("input-demo-001", ROOT / "artifacts/demo/consolidated")
assert list((ROOT / "artifacts/demo/consolidated/execution_id=input-demo-001").glob("*.parquet"))
con.close()
print("Consolidado y partición Parquet disponibles.")


Consolidado y partición Parquet disponibles.
